# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ansh-Saraswat-PC/FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip install duckdb scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 3.6 MB/s  0:00:04a 0:00:010:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 3.4 MB/s  0:00:02m 3.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 3.9 MB/s  0:00:05m 3.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [scikit-learn]0m 6/7 [scikit-learn]]
Note: you may need to restart the kernel to use updated packages.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The Contract:
What one row means: One row represents a single content item (page) on a specific day (Daily Grain).
Table used: fact_content_daily_performance (specifically the month=2026-03 partition for iteration).
Time window: March 1, 2026 to March 31, 2026.
Target/Proxy: A binary label predicting if the page will experience a >20% traffic drop in the following 30 days.
Excluded field: health_score. I am deliberately excluding this because it is a rule-based product decision made by FlyRank, not an observable search reality. If I include it, the model will just learn to reverse-engineer FlyRank's existing code instead of discovering true search signals.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions, clicks, position, content_age_days, sessions. (These are observable signals recorded daily).
Label: target_future_decline (A derived boolean based on the subsequent month's traffic).
Context: report_date, content_hash_id, client_hash_id. (Used for grouping and joining, never fed to the model).
Excluded: ga4_data_available. (A boolean system flag that doesn't measure performance, just tracking status).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Below, DuckDB connects to the Hugging Face warehouse to prove the data contract. It runs three verification queries (grain, span, availability), builds a 5-feature frame, and executes the leakage trap to prove why we must aggressively separate our labels from our features.

In [5]:
import duckdb
import pandas as pd
import os
import getpass
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 1. Setup & Auth
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    hf_token = getpass.getpass("HF_TOKEN not found in env. Paste your Hugging Face READ token here: ")

con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# THE FIX: Removed the non-existent "data/" folder from the path
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 2. Prove Three Facts
print("--- FACT 1: Grain Check (Date + Content ID) ---")
grain_check = con.sql(f"""
    SELECT report_date, content_hash_id, COUNT(*) as row_count
    FROM '{hf_path}'
    GROUP BY report_date, content_hash_id
    HAVING COUNT(*) > 1
""").df()
print(f"Rows violating daily grain (should be 0): {len(grain_check)}")

print("\n--- FACT 2: Row Count & Date Span ---")
span_check = con.sql(f"""
    SELECT MIN(report_date) as start_date, MAX(report_date) as end_date, COUNT(*) as total_rows
    FROM '{hf_path}'
""").df()
print(span_check.to_string(index=False))

print("\n--- FACT 3: Availability (IS TRUE) ---")
avail_check = con.sql(f"""
    SELECT COUNT(*) as rows_with_ga4_tracking
    FROM '{hf_path}'
    WHERE ga4_data_available IS TRUE
""").df()
print(avail_check.to_string(index=False))

# 3. Five Features & The Trap
print("\n--- THE LEAKAGE TRAP ---")
# Build a sample feature frame with a dummy target (e.g., highly visible page)
df_features = con.sql(f"""
    SELECT 
        gsc_impressions, 
        gsc_clicks, 
        gsc_avg_position as avg_position, 
        ga4_pageviews, 
        sessions_organic,
        CASE WHEN gsc_impressions > 1000 THEN 1 ELSE 0 END as target_high_vis,
        CASE WHEN gsc_impressions > 1000 THEN 'Yes' ELSE 'No' END as leaky_feature
    FROM '{hf_path}'
    WHERE gsc_impressions IS NOT NULL
    LIMIT 5000
""").df()

df_features['leaky_num'] = (df_features['leaky_feature'] == 'Yes').astype(int)

# 5 Honest features + 1 Leak
honest_features = ['gsc_impressions', 'gsc_clicks', 'avg_position', 'ga4_pageviews', 'sessions_organic']
X_leaky = df_features[honest_features + ['leaky_num']]
X_honest = df_features[honest_features]
y = df_features['target_high_vis']

X_train_l, X_test_l, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
X_train_h, X_test_h, _, _ = train_test_split(X_honest, y, test_size=0.3, random_state=42)

clf = DecisionTreeClassifier(max_depth=3)

clf.fit(X_train_l, y_train)
print(f"Accuracy WITH leaky target derivative: {accuracy_score(y_test, clf.predict(X_test_l)):.4f} (Too good to be true!)")

clf.fit(X_train_h, y_train)
print(f"Accuracy WITH honest features only: {accuracy_score(y_test, clf.predict(X_test_h)):.4f} (The real baseline)")

HF_TOKEN not found in env. Paste your Hugging Face READ token here:  ········


--- FACT 1: Grain Check (Date + Content ID) ---
Rows violating daily grain (should be 0): 0

--- FACT 2: Row Count & Date Span ---
start_date   end_date  total_rows
2026-03-01 2026-03-31     9841378

--- FACT 3: Availability (IS TRUE) ---
 rows_with_ga4_tracking
                 413966

--- THE LEAKAGE TRAP ---
Accuracy WITH leaky target derivative: 1.0000 (Too good to be true!)
Accuracy WITH honest features only: 1.0000 (The real baseline)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The Limitation: This dataset is an "unbalanced panel." Some clients did not connect their Google Analytics (GA4) accounts until late 2025. This means that if we blindly train a model on early 2025 data, it might interpret sessions = 0 as a dead page, when in reality it just means ga4_data_available = FALSE. We must explicitly check tracking start dates before defining our training windows.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.